In [ ]:
from __future__ import annotations

import json
import math
import os
import time
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

In [ ]:
# Config


@dataclass
class ToyKimiConfig:

    # Model dims
    d_model: int = 512
    n_heads: int = 8
    d_k: int = 64
    d_v: int = 64

    # Architecture
    n_blocks: int = 2
    kda_per_block: int = 3
    ffn_mult: float = 3.0

    # Sequence / chunking
    vocab_size: int = 50257
    max_seq_len: int = 1024
    chunk_size: int = 64

    # KDA-specific
    conv_kernel: int = 4
    gate_rank: int = 64

    # Regularization
    dropout: float = 0.0

    # Misc
    tie_embeddings: bool = True
    norm_eps: float = 1e-6

    @classmethod
    def full(cls, **overrides) -> "ToyKimiConfig":
        defaults = dict(
            d_model=512,
            n_heads=8,
            d_k=64,
            d_v=64,
            n_blocks=2,
            ffn_mult=3.0,
            gate_rank=64,
        )
        defaults.update(overrides)
        return cls(**defaults)

    @classmethod
    def half(cls, **overrides) -> "ToyKimiConfig":
        defaults = dict(
            d_model=384,
            n_heads=6,
            d_k=64,
            d_v=64,
            n_blocks=1,
            ffn_mult=3.0,
            gate_rank=64,
        )
        defaults.update(overrides)
        return cls(**defaults)

    @classmethod
    def quarter(cls, **overrides) -> "ToyKimiConfig":
        defaults = dict(
            d_model=256,
            n_heads=4,
            d_k=64,
            d_v=64,
            n_blocks=1,
            ffn_mult=2.5,
            gate_rank=64,
        )
        defaults.update(overrides)
        return cls(**defaults)

    @property
    def ffn_dim(self) -> int:
        return int(self.d_model * self.ffn_mult)

    @property
    def n_attn_layers(self) -> int:
        return self.n_blocks * (self.kda_per_block + 1)

    @property
    def n_kda_layers(self) -> int:
        return self.n_blocks * self.kda_per_block

    @property
    def n_full_attention_layers(self) -> int:
        return self.n_blocks

    def validate(self) -> None:
        assert self.d_model > 0
        assert self.n_heads > 0
        assert self.d_k > 0
        assert self.d_v > 0
        assert self.n_blocks > 0
        assert self.kda_per_block > 0
        assert self.ffn_dim > 0
        assert self.chunk_size > 0
        assert self.conv_kernel >= 1
        assert self.gate_rank > 0
        assert self.max_seq_len > 0
        assert self.vocab_size > 0


In [ ]:
# Core modules


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        dtype = x.dtype
        x_float = x.float()
        rms = torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        y = x_float * rms * self.weight.float()
        return y.to(dtype)


class ShortConv(nn.Module):
    # Causal depthwise 1D convolution used before q/k/v

    def __init__(self, channels: int, kernel_size: int = 4):
        super().__init__()
        self.channels = channels
        self.kernel_size = kernel_size
        self.pad_len = kernel_size - 1
        self.conv = nn.Conv1d(
            channels,
            channels,
            kernel_size,
            padding=0,
            groups=channels,
            bias=False,
        )

    def allocate_state(self, batch_size: int, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
        return torch.zeros(batch_size, self.channels, self.pad_len, device=device, dtype=dtype)

    def final_state_from_full_input(self, x: torch.Tensor) -> torch.Tensor:
        # Returns the raw pre-conv tail needed for recurrent decoding
        # x: [B, T, channels]
        # state: [B, channels, kernel_size - 1]

        B, T, C = x.shape
        if self.pad_len == 0:
            return x.new_zeros(B, C, 0)
        if T >= self.pad_len:
            return x[:, -self.pad_len :, :].transpose(1, 2).contiguous()
        pad = x.new_zeros(B, self.pad_len - T, C)
        return torch.cat([pad, x], dim=1).transpose(1, 2).contiguous()

    def forward_full(self, x: torch.Tensor) -> torch.Tensor:
        # Full causal convolution over a sequence. x: [B, T, C]
        xt = x.transpose(1, 2)
        xt = F.pad(xt, (self.pad_len, 0))
        return self.conv(xt).transpose(1, 2)

    def step(self, x_t: torch.Tensor, state: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # One-token recurrent convolution.

        xt = x_t.transpose(1, 2)  # [B, C, 1]
        if self.pad_len > 0:
            window = torch.cat([state, xt], dim=-1)
            new_state = window[:, :, 1:].contiguous()
        else:
            window = xt
            new_state = state

        weight = self.conv.weight[:, 0, :].to(window.dtype)  # [C, K]
        out = (window * weight.unsqueeze(0)).sum(dim=-1)
        if self.conv.bias is not None:
            out = out + self.conv.bias.to(out.dtype)
        return out.unsqueeze(1), new_state

    def forward(
        self,
        x: torch.Tensor,
        state: Optional[torch.Tensor] = None,
    ) -> tuple[torch.Tensor, Optional[torch.Tensor]]:
        if state is None:
            return self.forward_full(x), None
        return self.step(x, state)

In [ ]:
# KDA core

def _to_chunks(x: torch.Tensor, chunk_size: int) -> torch.Tensor:
    # [B, T, H, D] -> [B, H, N, C, D]. T must already be padded
    B, T, H, D = x.shape
    N = T // chunk_size
    return x.reshape(B, N, chunk_size, H, D).permute(0, 3, 1, 2, 4).contiguous()


def step_kda(
    q: torch.Tensor,          # [B, 1, H, dk]
    k: torch.Tensor,          # [B, 1, H, dk]
    v: torch.Tensor,          # [B, 1, H, dv]
    log_alpha: torch.Tensor,  # [B, 1, H, dk]
    beta: torch.Tensor,       # [B, 1, H, 1]
    state: torch.Tensor,      # [B, H, dk, dv]
    scale_q: bool = True,
) -> tuple[torch.Tensor, torch.Tensor]:
    # Single-step recurrent KDA matching
    q_t = q.squeeze(1)
    k_t = k.squeeze(1)
    v_t = v.squeeze(1)
    alpha = log_alpha.squeeze(1).exp()
    beta_t = beta.squeeze(1)

    # Diag(alpha_t) S_{t-1}
    S_pre = state.float() * alpha.float().unsqueeze(-1)

    # S_t = S_pre - beta * k * (k^T S_pre - v)^T
    kTS = torch.einsum("bhk,bhkv->bhv", k_t.float(), S_pre)
    correction = beta_t.float() * (kTS - v_t.float())
    S = S_pre - torch.einsum("bhk,bhv->bhkv", k_t.float(), correction)

    q_used = q_t.float() * (q.shape[-1] ** -0.5 if scale_q else 1.0)
    o_t = torch.einsum("bhkv,bhk->bhv", S, q_used)
    return o_t.unsqueeze(1).to(q.dtype), S.to(q.dtype)


@torch.no_grad()
def kda_recurrent_reference(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    log_alpha: torch.Tensor,
    beta: torch.Tensor,
    initial_state: Optional[torch.Tensor] = None,
    scale_q: bool = True,
) -> tuple[torch.Tensor, torch.Tensor]:
    # Slow recurrent reference for correctness tests
    B, T, H, dk = q.shape
    dv = v.shape[-1]
    if initial_state is None:
        state = torch.zeros(B, H, dk, dv, device=q.device, dtype=q.dtype)
    else:
        state = initial_state
    outs = []
    for t in range(T):
        o_t, state = step_kda(
            q[:, t : t + 1],
            k[:, t : t + 1],
            v[:, t : t + 1],
            log_alpha[:, t : t + 1],
            beta[:, t : t + 1],
            state,
            scale_q=scale_q,
        )
        outs.append(o_t)
    return torch.cat(outs, dim=1), state


def chunk_kda(
    q: torch.Tensor,          # [B, T, H, dk]
    k: torch.Tensor,          # [B, T, H, dk]
    v: torch.Tensor,          # [B, T, H, dv]
    log_alpha: torch.Tensor,  # [B, T, H, dk]
    beta: torch.Tensor,       # [B, T, H, 1]
    chunk_size: int,
    initial_state: Optional[torch.Tensor] = None,
    scale_q: bool = True,
) -> tuple[torch.Tensor, torch.Tensor]:
    # Chunkwise KDA implementation following the paper's PyTorch-style algorithm
    #The implementation is intentionally explicit and tested against the recurrent reference

    orig_dtype = q.dtype
    B, T_orig, H, dk = q.shape
    dv = v.shape[-1]
    C = chunk_size

    assert k.shape == (B, T_orig, H, dk)
    assert v.shape == (B, T_orig, H, dv)
    assert log_alpha.shape == (B, T_orig, H, dk)
    assert beta.shape == (B, T_orig, H, 1)

    pad = (C - T_orig % C) % C
    if pad > 0:
        q = F.pad(q, (0, 0, 0, 0, 0, pad))
        k = F.pad(k, (0, 0, 0, 0, 0, pad))
        v = F.pad(v, (0, 0, 0, 0, 0, pad))
        log_alpha = F.pad(log_alpha, (0, 0, 0, 0, 0, pad))
        beta = F.pad(beta, (0, 0, 0, 0, 0, pad), value=0.0)

    T = T_orig + pad
    N_chunks = T // C

    q_c = _to_chunks(q, C)                                  # [B,H,N,C,dk]
    k_c = _to_chunks(k, C)
    v_c = _to_chunks(v, C)
    g = _to_chunks(log_alpha, C).float()                     # [B,H,N,C,dk]
    beta_c = _to_chunks(beta, C).float().squeeze(-1)         # [B,H,N,C]

    if scale_q:
        q_c = q_c * (dk ** -0.5)

    gc = g.cumsum(dim=3)                                     # cumulative log decay inside chunk

    mask_lower = torch.tril(torch.ones(C, C, device=q.device, dtype=torch.bool), diagonal=-1)
    mask_causal = torch.tril(torch.ones(C, C, device=q.device, dtype=torch.bool), diagonal=0)
    zero = torch.tensor(0.0, device=q.device, dtype=torch.float32)

    # KDA uses products of cumulative decays and their inverse. Clamp inverse side to avoid inf in early random training.
    g_exp = gc.exp()
    neg_g_exp = (-gc).clamp(max=80.0).exp()

    # Akk[row=i, col=j] = <k_i * gamma_i, k_j / gamma_j>, strict lower later.
    kp = k_c * g_exp.to(k_c.dtype)
    km = k_c * neg_g_exp.to(k_c.dtype)
    Akk = torch.matmul(kp, km.transpose(-1, -2)).float()
    Akk = torch.where(mask_lower, Akk, zero)

    # M = (I + StrictTril(Diag(beta) Akk))^{-1} Diag(beta)
    # forward-substitution on a strictly-lower triangular matrix
    M = -(Akk * beta_c.unsqueeze(-1))
    for i in range(1, C):
        M[..., i, :i] = M[..., i, :i].clone() + (
            M[..., i, :, None].clone() * M[..., :, :i].clone()
        ).sum(dim=-2)
    eye = torch.eye(C, device=q.device, dtype=torch.float32)
    M = M + eye
    M = M * beta_c.unsqueeze(-2)

    W = torch.matmul(M.to(k_c.dtype), kp).float()            # [B,H,N,C,dk]
    U = torch.matmul(M.to(v_c.dtype), v_c).float()           # [B,H,N,C,dv]

    if initial_state is None:
        S = torch.zeros(B, H, dk, dv, device=q.device, dtype=torch.float32)
    else:
        S = initial_state.float()

    outputs = []
    qp = q_c * g_exp.to(q_c.dtype)

    for n in range(N_chunks):
        qpn = qp[:, :, n]
        kmn = km[:, :, n]
        kn = k_c[:, :, n].float()
        gn = gc[:, :, n]
        Wn = W[:, :, n]
        Un = U[:, :, n]

        Aqk = torch.matmul(qpn, kmn.transpose(-1, -2)).float()
        Aqk = torch.where(mask_causal, Aqk, zero)

        # Pseudo-value term U - W S.
        pseudo_v = Un - torch.einsum("bhck,bhkv->bhcv", Wn, S)
        inter = torch.einsum("bhck,bhkv->bhcv", qpn.float(), S)
        intra = torch.matmul(Aqk, pseudo_v)
        outputs.append(inter + intra)

        # Chunk state update.
        g_last = gn[:, :, -1:]                              # [B,H,1,dk]
        S = S * g_last.squeeze(2).exp().unsqueeze(-1)
        k_decayed = (g_last - gn).exp() * kn
        S = S + torch.einsum("bhck,bhcv->bhkv", k_decayed, pseudo_v)

    out = torch.stack(outputs, dim=2)                        # [B,H,N,C,dv]
    out = out.reshape(B, H, T, dv).permute(0, 2, 1, 3).contiguous()
    if pad > 0:
        out = out[:, :T_orig]
    return out.to(orig_dtype), S.to(orig_dtype)

In [ ]:
# Caches


@dataclass
class KDALayerCache:
    state: torch.Tensor      # [B, H, dk, dv]
    conv_q: torch.Tensor     # [B, H*dk, K-1]
    conv_k: torch.Tensor     # [B, H*dk, K-1]
    conv_v: torch.Tensor     # [B, H*dv, K-1]


@dataclass
class MLACache:
    k: torch.Tensor          # [B, H, T, dk]
    v: torch.Tensor          # [B, H, T, dv]


@dataclass
class BlockCache:
    kda_caches: list[KDALayerCache]
    mla_cache: Optional[MLACache]

In [ ]:
# Layers


class KDALayer(nn.Module):
    def __init__(self, cfg: ToyKimiConfig):
        super().__init__()
        cfg.validate()
        d, h, dk, dv = cfg.d_model, cfg.n_heads, cfg.d_k, cfg.d_v
        self.cfg = cfg
        self.h = h
        self.dk = dk
        self.dv = dv

        self.W_q = nn.Linear(d, h * dk, bias=False)
        self.W_k = nn.Linear(d, h * dk, bias=False)
        self.W_v = nn.Linear(d, h * dv, bias=False)

        self.conv_q = ShortConv(h * dk, cfg.conv_kernel)
        self.conv_k = ShortConv(h * dk, cfg.conv_kernel)
        self.conv_v = ShortConv(h * dv, cfg.conv_kernel)

        self.W_alpha_down = nn.Linear(d, cfg.gate_rank, bias=False)
        self.W_alpha_up = nn.Linear(cfg.gate_rank, h * dk, bias=False)
        self.W_beta = nn.Linear(d, h, bias=True)

        self.W_g_down = nn.Linear(d, cfg.gate_rank, bias=False)
        self.W_g_up = nn.Linear(cfg.gate_rank, h * dv, bias=False)

        self.head_norm = RMSNorm(dv, eps=cfg.norm_eps)
        self.dropout = nn.Dropout(cfg.dropout)
        self.W_o = nn.Linear(h * dv, d, bias=False)

    def allocate_cache(self, batch_size: int, device: torch.device, dtype: torch.dtype) -> KDALayerCache:
        h, dk, dv = self.h, self.dk, self.dv
        return KDALayerCache(
            state=torch.zeros(batch_size, h, dk, dv, device=device, dtype=dtype),
            conv_q=self.conv_q.allocate_state(batch_size, device, dtype),
            conv_k=self.conv_k.allocate_state(batch_size, device, dtype),
            conv_v=self.conv_v.allocate_state(batch_size, device, dtype),
        )

    def _params_from_conv_outputs(
        self,
        x: torch.Tensor,
        q_conv: torch.Tensor,
        k_conv: torch.Tensor,
        v_conv: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        B, T, _ = x.shape
        h, dk, dv = self.h, self.dk, self.dv

        q = F.silu(q_conv).reshape(B, T, h, dk)
        k = F.silu(k_conv).reshape(B, T, h, dk)
        v = F.silu(v_conv).reshape(B, T, h, dv)

        q = F.normalize(q, p=2, dim=-1, eps=1e-8)
        k = F.normalize(k, p=2, dim=-1, eps=1e-8)

        # log alpha in [-10, 0]. exp(log_alpha) in [~4.5e-5, 1].
        log_alpha = F.logsigmoid(self.W_alpha_up(self.W_alpha_down(x)))
        log_alpha = log_alpha.reshape(B, T, h, dk).clamp(min=-10.0)
        beta = torch.sigmoid(self.W_beta(x)).reshape(B, T, h, 1)
        return q, k, v, log_alpha, beta

    def _finish_output(self, x: torch.Tensor, o: torch.Tensor) -> torch.Tensor:
        B, T, h, dv = o.shape
        o = self.head_norm(o)
        gate = torch.sigmoid(self.W_g_up(self.W_g_down(x))).reshape(B, T, h, dv)
        o = gate * o
        o = self.dropout(o)
        return self.W_o(o.reshape(B, T, h * dv))

    def forward(
        self,
        x: torch.Tensor,
        cache: Optional[KDALayerCache] = None,
    ) -> tuple[torch.Tensor, Optional[KDALayerCache]]:
        # Full sequence if cache is None; one-token recurrent step if cache is given

        B, T, _ = x.shape
        q_raw = self.W_q(x)
        k_raw = self.W_k(x)
        v_raw = self.W_v(x)

        if cache is None:
            q_conv = self.conv_q.forward_full(q_raw)
            k_conv = self.conv_k.forward_full(k_raw)
            v_conv = self.conv_v.forward_full(v_raw)
            q, k, v, log_alpha, beta = self._params_from_conv_outputs(x, q_conv, k_conv, v_conv)
            o, _ = chunk_kda(q, k, v, log_alpha, beta, self.cfg.chunk_size)
            return self._finish_output(x, o), None

        assert T == 1, "Cached KDA forward is intended for one-token recurrent decoding."
        q_conv, new_cq = self.conv_q.step(q_raw, cache.conv_q)
        k_conv, new_ck = self.conv_k.step(k_raw, cache.conv_k)
        v_conv, new_cv = self.conv_v.step(v_raw, cache.conv_v)
        q, k, v, log_alpha, beta = self._params_from_conv_outputs(x, q_conv, k_conv, v_conv)
        o, new_state = step_kda(q, k, v, log_alpha, beta, cache.state)
        new_cache = KDALayerCache(new_state, new_cq, new_ck, new_cv)
        return self._finish_output(x, o), new_cache

    def forward_with_cache(self, x: torch.Tensor) -> tuple[torch.Tensor, KDALayerCache]:
        # Parallel prefill that also extracts the recurrent cache after the sequence
        q_raw = self.W_q(x)
        k_raw = self.W_k(x)
        v_raw = self.W_v(x)

        q_conv = self.conv_q.forward_full(q_raw)
        k_conv = self.conv_k.forward_full(k_raw)
        v_conv = self.conv_v.forward_full(v_raw)

        q, k, v, log_alpha, beta = self._params_from_conv_outputs(x, q_conv, k_conv, v_conv)
        o, final_state = chunk_kda(q, k, v, log_alpha, beta, self.cfg.chunk_size)
        out = self._finish_output(x, o)

        cache = KDALayerCache(
            state=final_state.contiguous(),
            conv_q=self.conv_q.final_state_from_full_input(q_raw),
            conv_k=self.conv_k.final_state_from_full_input(k_raw),
            conv_v=self.conv_v.final_state_from_full_input(v_raw),
        )
        return out, cache


class NoPEMHALayer(nn.Module):
    # NoPE full multi-head attention
    # It is not true latent-compression MLA implementation

    def __init__(self, cfg: ToyKimiConfig):
        super().__init__()
        d, h, dk, dv = cfg.d_model, cfg.n_heads, cfg.d_k, cfg.d_v
        self.h = h
        self.dk = dk
        self.dv = dv
        self.W_q = nn.Linear(d, h * dk, bias=False)
        self.W_k = nn.Linear(d, h * dk, bias=False)
        self.W_v = nn.Linear(d, h * dv, bias=False)
        self.W_o = nn.Linear(h * dv, d, bias=False)
        self.dropout_p = cfg.dropout

    def _qkv(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        B, T, _ = x.shape
        q = self.W_q(x).reshape(B, T, self.h, self.dk).transpose(1, 2)
        k = self.W_k(x).reshape(B, T, self.h, self.dk).transpose(1, 2)
        v = self.W_v(x).reshape(B, T, self.h, self.dv).transpose(1, 2)
        return q, k, v

    def forward(
        self,
        x: torch.Tensor,
        cache: Optional[MLACache] = None,
    ) -> tuple[torch.Tensor, MLACache]:
        B, T, _ = x.shape
        q, k_new, v_new = self._qkv(x)

        if cache is None:
            k = k_new
            v = v_new
            is_causal = True
        else:
            k = torch.cat([cache.k, k_new], dim=2)
            v = torch.cat([cache.v, v_new], dim=2)
            is_causal = False

        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=is_causal,
        )
        out = self.W_o(attn_out.transpose(1, 2).reshape(B, T, self.h * self.dv))
        return out, MLACache(k=k, v=v)

    def forward_with_cache(self, x: torch.Tensor) -> tuple[torch.Tensor, MLACache]:
        return self.forward(x, cache=None)


class SwiGLUFFN(nn.Module):
    def __init__(self, cfg: ToyKimiConfig):
        super().__init__()
        d, ffn = cfg.d_model, cfg.ffn_dim
        self.W_gate = nn.Linear(d, ffn, bias=False)
        self.W_up = nn.Linear(d, ffn, bias=False)
        self.W_down = nn.Linear(ffn, d, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.silu(self.W_gate(x)) * self.W_up(x)
        x = self.dropout(x)
        return self.W_down(x)


class KimiLinearBlock(nn.Module):
    # Macro block: [KDA + FFN] x 3, then [NoPE full attention + FFN] x 1

    def __init__(self, cfg: ToyKimiConfig):
        super().__init__()
        self.kda_layers = nn.ModuleList([KDALayer(cfg) for _ in range(cfg.kda_per_block)])
        self.kda_norms = nn.ModuleList([RMSNorm(cfg.d_model, cfg.norm_eps) for _ in range(cfg.kda_per_block)])
        self.kda_ffns = nn.ModuleList([SwiGLUFFN(cfg) for _ in range(cfg.kda_per_block)])
        self.kda_ffn_norms = nn.ModuleList([RMSNorm(cfg.d_model, cfg.norm_eps) for _ in range(cfg.kda_per_block)])

        self.full_attn = NoPEMHALayer(cfg)
        self.full_attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.full_ffn = SwiGLUFFN(cfg)
        self.full_ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)

    def allocate_cache(self, batch_size: int, device: torch.device, dtype: torch.dtype) -> BlockCache:
        return BlockCache(
            kda_caches=[layer.allocate_cache(batch_size, device, dtype) for layer in self.kda_layers],
            mla_cache=None,
        )

    def forward(self, x: torch.Tensor, cache: Optional[BlockCache] = None) -> tuple[torch.Tensor, Optional[BlockCache]]:
        new_kda_caches: list[KDALayerCache] = []

        for i, (kda, norm, ffn, ffn_norm) in enumerate(
            zip(self.kda_layers, self.kda_norms, self.kda_ffns, self.kda_ffn_norms)
        ):
            layer_cache = cache.kda_caches[i] if cache is not None else None
            y, new_layer_cache = kda(norm(x), cache=layer_cache)
            x = x + y
            if cache is not None:
                assert new_layer_cache is not None
                new_kda_caches.append(new_layer_cache)
            x = x + ffn(ffn_norm(x))

        mla_cache = cache.mla_cache if cache is not None else None
        y, new_mla_cache = self.full_attn(self.full_attn_norm(x), cache=mla_cache)
        x = x + y
        x = x + self.full_ffn(self.full_ffn_norm(x))

        if cache is None:
            return x, None
        return x, BlockCache(kda_caches=new_kda_caches, mla_cache=new_mla_cache)

    def forward_with_cache(self, x: torch.Tensor) -> tuple[torch.Tensor, BlockCache]:
        caches: list[KDALayerCache] = []
        for kda, norm, ffn, ffn_norm in zip(
            self.kda_layers, self.kda_norms, self.kda_ffns, self.kda_ffn_norms
        ):
            y, layer_cache = kda.forward_with_cache(norm(x))
            x = x + y
            caches.append(layer_cache)
            x = x + ffn(ffn_norm(x))

        y, mla_cache = self.full_attn.forward_with_cache(self.full_attn_norm(x))
        x = x + y
        x = x + self.full_ffn(self.full_ffn_norm(x))
        return x, BlockCache(kda_caches=caches, mla_cache=mla_cache)

In [ ]:
# Full model


class ToyKimiLinear(nn.Module):
    def __init__(self, cfg: ToyKimiConfig):
        super().__init__()
        cfg.validate()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([KimiLinearBlock(cfg) for _ in range(cfg.n_blocks)])
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        self.apply(self._init_weights)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

    def _init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    def allocate_cache(self, batch_size: int, device: torch.device, dtype: torch.dtype) -> list[BlockCache]:
        return [block.allocate_cache(batch_size, device, dtype) for block in self.blocks]

    def forward(
        self,
        input_ids: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
        caches: Optional[list[BlockCache]] = None,
    ) -> tuple[torch.Tensor, Optional[torch.Tensor], Optional[list[BlockCache]]]:
        x = self.tok_emb(input_ids)
        x = self.drop(x)

        new_caches: Optional[list[BlockCache]] = [] if caches is not None else None
        for i, block in enumerate(self.blocks):
            block_cache = caches[i] if caches is not None else None
            x, new_block_cache = block(x, cache=block_cache)
            if new_caches is not None:
                assert new_block_cache is not None
                new_caches.append(new_block_cache)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, new_caches

    @torch.no_grad()
    def prefill(self, input_ids: torch.Tensor) -> tuple[torch.Tensor, list[BlockCache]]:
        """Full-sequence prefill that extracts caches for recurrent decoding."""
        x = self.tok_emb(input_ids)
        x = self.drop(x)
        caches: list[BlockCache] = []
        for block in self.blocks:
            x, cache = block.forward_with_cache(x)
            caches.append(cache)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits, caches

    @torch.no_grad()
    def step(self, input_id_t: torch.Tensor, caches: list[BlockCache]) -> tuple[torch.Tensor, list[BlockCache]]:
        """One-token recurrent decode step. input_id_t: [B]."""
        logits, _, new_caches = self.forward(input_id_t[:, None], targets=None, caches=caches)
        assert new_caches is not None
        return logits[:, -1, :], new_caches

    @torch.no_grad()
    def generate_full_prefix(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int = 128,
        temperature: float = 0.8,
        top_k: int = 40,
    ) -> torch.Tensor:
        self.eval()
        out = input_ids
        for _ in range(max_new_tokens):
            idx = out[:, -self.cfg.max_seq_len :]
            logits, _, _ = self.forward(idx)
            next_logits = logits[:, -1, :] / max(temperature, 1e-8)
            if top_k > 0:
                values, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
                next_logits = next_logits.masked_fill(next_logits < values[:, [-1]], -float("inf"))
            probs = F.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            out = torch.cat([out, next_id], dim=1)
        return out

    @torch.no_grad()
    def generate_recurrent(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int = 128,
        temperature: float = 0.8,
        top_k: int = 40,
        optimized_prefill: bool = True,
    ) -> torch.Tensor:
        self.eval()
        B = input_ids.size(0)
        device = input_ids.device
        dtype = self.tok_emb.weight.dtype

        if optimized_prefill:
            prefill_logits, caches = self.prefill(input_ids)
            logits = prefill_logits[:, -1, :]
        else:
            caches = self.allocate_cache(B, device, dtype)
            logits = None
            for t in range(input_ids.size(1)):
                logits, caches = self.step(input_ids[:, t], caches)
            assert logits is not None

        out = input_ids
        for _ in range(max_new_tokens):
            next_logits = logits / max(temperature, 1e-8)
            if top_k > 0:
                values, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
                next_logits = next_logits.masked_fill(next_logits < values[:, [-1]], -float("inf"))
            probs = F.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).squeeze(-1)
            out = torch.cat([out, next_id[:, None]], dim=1)
            logits, caches = self.step(next_id, caches)
        return out

    @torch.no_grad()
    def generate(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int = 128,
        temperature: float = 0.8,
        top_k: int = 40,
        use_recurrent: bool = True,
        optimized_prefill: bool = True,
    ) -> torch.Tensor:
        if use_recurrent:
            return self.generate_recurrent(
                input_ids,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_k=top_k,
                optimized_prefill=optimized_prefill,
            )
        return self.generate_full_prefix(input_ids, max_new_tokens, temperature, top_k)

    def num_parameters(self, only_trainable: bool = True) -> int:
        return sum(
            p.numel() for p in self.parameters() if (not only_trainable or p.requires_grad)
        )

In [ ]:
# Correctness tests


def test_chunk_kda_matches_recurrent(device: torch.device) -> None:
    torch.manual_seed(0)
    cases = [
        dict(T=1, C=4),
        dict(T=7, C=4),
        dict(T=16, C=8),
        dict(T=17, C=8),
        dict(T=33, C=16),
    ]
    for case in cases:
        B, T, H, dk, dv = 2, case["T"], 3, 5, 7
        C = case["C"]
        q = F.normalize(torch.randn(B, T, H, dk, device=device), dim=-1)
        k = F.normalize(torch.randn(B, T, H, dk, device=device), dim=-1)
        v = torch.randn(B, T, H, dv, device=device)
        # Keep alpha away from exactly 0 or 1 for a nontrivial test.
        log_alpha = torch.log(torch.empty(B, T, H, dk, device=device).uniform_(0.75, 0.99))
        beta = torch.empty(B, T, H, 1, device=device).uniform_(0.05, 0.9)

        y_chunk, s_chunk = chunk_kda(q, k, v, log_alpha, beta, C)
        y_rec, s_rec = kda_recurrent_reference(q, k, v, log_alpha, beta)

        y_err = (y_chunk.float() - y_rec.float()).abs().max().item()
        s_err = (s_chunk.float() - s_rec.float()).abs().max().item()
        tol = 2e-4
        if y_err > tol or s_err > tol:
            raise AssertionError(
                f"KDA chunk/recurrent mismatch T={T}, C={C}: "
                f"y_err={y_err:.6g}, s_err={s_err:.6g}, tol={tol:.6g}"
            )
    print("KDA chunkwise/recurrent correctness test passed.")


@torch.no_grad()
def test_model_recurrent_matches_full(device: torch.device) -> None:
    torch.manual_seed(1)
    cfg = ToyKimiConfig.quarter(max_seq_len=64, chunk_size=8, dropout=0.0)
    model = ToyKimiLinear(cfg).to(device)
    model.eval()

    input_ids = torch.randint(0, cfg.vocab_size, (2, 17), device=device)
    logits_full, _, _ = model(input_ids)

    caches = model.allocate_cache(
        batch_size=input_ids.size(0),
        device=device,
        dtype=model.tok_emb.weight.dtype,
    )
    logits_step = None
    for t in range(input_ids.size(1)):
        logits_step, caches = model.step(input_ids[:, t], caches)

    assert logits_step is not None
    err = (logits_full[:, -1].float() - logits_step.float()).abs().max().item()
    tol = 1e-3
    if err > tol:
        raise AssertionError(f"Model recurrent/full mismatch: err={err:.6g}, tol={tol:.6g}")
    print("Model step-by-step recurrent path matches full forward test passed.")


@torch.no_grad()
def test_optimized_prefill_matches_full_and_step(device: torch.device) -> None:
    torch.manual_seed(2)
    cfg = ToyKimiConfig.quarter(max_seq_len=64, chunk_size=8, dropout=0.0)
    model = ToyKimiLinear(cfg).to(device)
    model.eval()

    prompt = torch.randint(0, cfg.vocab_size, (2, 19), device=device)
    next_token = torch.randint(0, cfg.vocab_size, (2,), device=device)
    prompt_plus_one = torch.cat([prompt, next_token[:, None]], dim=1)

    logits_full_prompt, _, _ = model(prompt)
    logits_prefill, caches = model.prefill(prompt)
    err_prefill = (logits_full_prompt.float() - logits_prefill.float()).abs().max().item()
    tol = 1e-3
    if err_prefill > tol:
        raise AssertionError(f"Optimized prefill logits mismatch: err={err_prefill:.6g}, tol={tol}")

    logits_full_plus_one, _, _ = model(prompt_plus_one)
    logits_step, _ = model.step(next_token, caches)
    err_next = (logits_full_plus_one[:, -1].float() - logits_step.float()).abs().max().item()
    if err_next > tol:
        raise AssertionError(f"Optimized prefill + step mismatch: err={err_next:.6g}, tol={tol}")

    print("Optimized prefill cache test passed.")


def run_correctness_tests() -> None:
    # CPU fp32 gives the cleanest deterministic signal
    device = torch.device("cpu")
    test_chunk_kda_matches_recurrent(device)
    test_model_recurrent_matches_full(device)
    test_optimized_prefill_matches_full_and_step(device)

In [ ]:
# Data Preprocessing and Handling


class PackedTokenDataset(Dataset):
    def __init__(self, token_ids: list[int], seq_len: int):
        if len(token_ids) < seq_len + 1:
            raise ValueError(f"Need at least {seq_len + 1} tokens, got {len(token_ids)}.")
        self.data = torch.tensor(token_ids, dtype=torch.long)
        self.seq_len = seq_len
        self.n_samples = (len(self.data) - 1) // seq_len

    def __len__(self) -> int:
        return self.n_samples

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        start = idx * self.seq_len
        chunk = self.data[start : start + self.seq_len + 1]
        return chunk[:-1], chunk[1:]


def _row_to_text(row: dict) -> str:
    for key in ("text", "content", "story"):
        value = row.get(key)
        if isinstance(value, str):
            return value
    return ""


def load_tokens(
    dataset_name: str,
    tokenizer,
    split: str = "train",
    max_tokens: Optional[int] = None,
) -> list[int]:
    from datasets import load_dataset

    if dataset_name.startswith("file:"):
        path = dataset_name[5:]
        with open(path, encoding="utf-8") as f:
            text = f.read()
        tokens = tokenizer.encode(text)
        return tokens[:max_tokens] if max_tokens else tokens

    ds_configs = {
        "wikitext-2": ("wikitext", "wikitext-2-raw-v1", False),
        "wikitext-103": ("wikitext", "wikitext-103-raw-v1", False),
        "tinystories": ("roneneldan/TinyStories", None, False),
        "openwebtext": ("Skylion007/openwebtext", None, True),
        "fineweb-edu": ("HuggingFaceFW/fineweb-edu-score-2", "sample-10BT", True),
        "slimpajama": ("cerebras/SlimPajama-627B", None, True),
        "c4": ("allenai/c4", "en", True),
    }
    if dataset_name not in ds_configs:
        raise ValueError(f"Unknown dataset '{dataset_name}'.")

    ds_name, ds_config, streaming = ds_configs[dataset_name]
    limit = max_tokens or 500_000_000
    tokens: list[int] = []

    if streaming:
        ds = load_dataset(ds_name, ds_config, split=split, streaming=True)
        for row in ds:
            text = _row_to_text(row)
            if text.strip():
                tokens.extend(tokenizer.encode(text))
                if len(tokens) >= limit:
                    tokens = tokens[:limit]
                    break
    else:
        ds = load_dataset(ds_name, ds_config, split=split)
        for row in ds:
            text = _row_to_text(row)
            if text.strip():
                tokens.extend(tokenizer.encode(text))
                if len(tokens) >= limit:
                    tokens = tokens[:limit]
                    break
    return tokens


def make_data_splits(
    dataset_name: str,
    tokenizer,
    max_tokens: Optional[int],
    val_monitor_fraction: float,
    val_ablation_fraction: float,
    test_replication_fraction: float,
    val_monitor_max_tokens: Optional[int],
    val_ablation_max_tokens: Optional[int],
    test_replication_max_tokens: Optional[int],
) -> tuple[dict[str, list[int]], dict]:
    def cap(tokens: list[int], cap_n: Optional[int]) -> list[int]:
        return tokens[:cap_n] if cap_n is not None else tokens

    info = {
        "dataset": dataset_name,
        "fractions": {
            "val_monitor": val_monitor_fraction,
            "val_ablation": val_ablation_fraction,
            "test_replication": test_replication_fraction,
        },
    }

    if dataset_name in {"wikitext-2", "wikitext-103"}:
        train_all = load_tokens(dataset_name, tokenizer, split="train", max_tokens=max_tokens)
        val_monitor = load_tokens(dataset_name, tokenizer, split="validation", max_tokens=val_monitor_max_tokens)
        test_replication = load_tokens(dataset_name, tokenizer, split="test", max_tokens=test_replication_max_tokens)
        n_ablation = max(int(len(train_all) * val_ablation_fraction), 1)
        if val_ablation_max_tokens is not None:
            n_ablation = min(n_ablation, val_ablation_max_tokens)
        train = train_all[:-n_ablation]
        val_ablation = train_all[-n_ablation:]
        info["split_policy"] = "official validation/test; val_ablation carved from train tail"
    else:
        all_tokens = load_tokens(dataset_name, tokenizer, split="train", max_tokens=max_tokens)
        n_total = len(all_tokens)
        n_monitor = max(int(n_total * val_monitor_fraction), 1)
        n_ablation = max(int(n_total * val_ablation_fraction), 1)
        n_test = max(int(n_total * test_replication_fraction), 1)
        if val_monitor_max_tokens is not None:
            n_monitor = min(n_monitor, val_monitor_max_tokens)
        if val_ablation_max_tokens is not None:
            n_ablation = min(n_ablation, val_ablation_max_tokens)
        if test_replication_max_tokens is not None:
            n_test = min(n_test, test_replication_max_tokens)

        held = n_monitor + n_ablation + n_test
        if n_total - held < 2:
            raise ValueError("Not enough tokens after held-out split.")
        train_end = n_total - held
        monitor_end = train_end + n_monitor
        ablation_end = monitor_end + n_ablation
        train = all_tokens[:train_end]
        val_monitor = all_tokens[train_end:monitor_end]
        val_ablation = all_tokens[monitor_end:ablation_end]
        test_replication = all_tokens[ablation_end:]
        info["split_policy"] = "tail holdout from train token stream"
        info["n_total_loaded_tokens"] = n_total

    splits = {
        "train": train,
        "val_monitor": cap(val_monitor, val_monitor_max_tokens),
        "val_ablation": cap(val_ablation, val_ablation_max_tokens),
        "test_replication": cap(test_replication, test_replication_max_tokens),
    }
    info["n_tokens"] = {k: len(v) for k, v in splits.items()}
    return splits, info


def verify_splits(splits: dict[str, list[int]], seq_len: int) -> dict:
    checks = {}
    for name in ["train", "val_monitor", "val_ablation", "test_replication"]:
        if len(splits[name]) < seq_len + 1:
            raise ValueError(f"Split {name} too small: {len(splits[name])} tokens.")
        checks[name] = {
            "tokens": len(splits[name]),
            "samples": (len(splits[name]) - 1) // seq_len,
        }
    checks["windows_cross_boundary"] = False
    checks["note"] = "Each split gets its own PackedTokenDataset; no packed window crosses split boundaries."
    return checks

In [ ]:
# Training helpers

class TrainingLogger:
    def __init__(self, log_path: str, run_config: dict):
        self.log_path = Path(log_path)
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        self.file = open(self.log_path, "w", encoding="utf-8")
        self.write({"type": "run_config", "timestamp": datetime.now().isoformat(), **run_config})

    def write(self, payload: dict) -> None:
        self.file.write(json.dumps(payload, default=str) + "\n")
        self.file.flush()

    def log_step(self, metrics: dict) -> None:
        self.write({"type": "step", "timestamp": datetime.now().isoformat(), **metrics})

    def log_eval(self, metrics: dict) -> None:
        self.write({"type": "eval", "timestamp": datetime.now().isoformat(), **metrics})

    def log_final(self, metrics: dict) -> None:
        self.write({"type": "final", "timestamp": datetime.now().isoformat(), **metrics})

    def close(self) -> None:
        self.file.close()


def cosine_with_warmup(step: int, warmup: int, total: int) -> float:
    if step < warmup:
        return step / max(warmup, 1)
    progress = (step - warmup) / max(total - warmup, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


def gpu_stats(device: torch.device) -> dict:
    if device.type != "cuda":
        return {}
    return {
        "gpu_mem_allocated_gb": round(torch.cuda.memory_allocated(device) / 1e9, 3),
        "gpu_mem_reserved_gb": round(torch.cuda.memory_reserved(device) / 1e9, 3),
        "gpu_mem_peak_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 3),
    }


def estimate_flops_per_optimizer_step(cfg: ToyKimiConfig, effective_batch_size: int) -> dict:
    # Rough logging estimate, not an exact MFU computation
    T = cfg.max_seq_len
    C = cfg.chunk_size
    d = cfg.d_model
    dk = cfg.d_k
    dv = cfg.d_v
    h = cfg.n_heads
    n_kda = cfg.n_kda_layers
    n_mha = cfg.n_full_attention_layers
    ffn = cfg.ffn_dim

    flops_kda_head = 6 * T * dk**2 + 3 * T * C * dk + T * C**2
    flops_kda = n_kda * h * flops_kda_head

    # QK^T + AttnV for full attention.
    flops_mha_head = 2 * T * T * dk + 2 * T * T * dv
    flops_mha = n_mha * h * flops_mha_head

    flops_proj_kda_layer = T * (3 * 2 * d * h * dk + 2 * h * dv * d)
    flops_proj_mha_layer = T * (3 * 2 * d * h * dk + 2 * h * dv * d)
    flops_proj = n_kda * flops_proj_kda_layer + n_mha * flops_proj_mha_layer

    # SwiGLU gate + up + down per token-mixing layer.
    n_ffn = n_kda + n_mha
    flops_ffn = n_ffn * (3 * 2 * T * d * ffn)

    # Embedding lookup ignored; LM head matmul counted.
    flops_lm_head = 2 * T * d * cfg.vocab_size

    fwd = flops_kda + flops_mha + flops_proj + flops_ffn + flops_lm_head
    step = 3 * fwd * effective_batch_size
    return {
        "estimated_flops_fwd_per_sample": fwd,
        "estimated_flops_per_optimizer_step": step,
        "breakdown": {
            "kda_attention": flops_kda,
            "nope_mha_attention": flops_mha,
            "projections": flops_proj,
            "ffn": flops_ffn,
            "lm_head": flops_lm_head,
        },
    }


def make_optimizer(model: nn.Module, lr: float, weight_decay: float) -> torch.optim.Optimizer:
    decay, no_decay = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        lower = name.lower()
        if name.endswith("bias") or "norm" in lower or "w_beta" in lower:
            no_decay.append(param)
        else:
            decay.append(param)
    return torch.optim.AdamW(
        [
            {"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0},
        ],
        lr=lr,
        betas=(0.9, 0.95),
        fused=torch.cuda.is_available(),
    )


@torch.no_grad()
def evaluate(
    model: ToyKimiLinear,
    loader: DataLoader,
    device: torch.device,
    amp_dtype: torch.dtype,
    max_batches: int,
    use_amp: bool,
) -> dict:
    model.eval()
    losses = []
    n_tokens = 0
    t0 = time.time()
    for i, (x, y) in enumerate(loader):
        if i >= max_batches:
            break
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with torch.amp.autocast(device.type, dtype=amp_dtype, enabled=use_amp):
            _, loss, _ = model(x, y)
        losses.append(float(loss.item()))
        n_tokens += x.numel()
    if not losses:
        raise RuntimeError("Evaluation loader yielded no batches.")
    val_loss = sum(losses) / len(losses)
    model.train()
    return {
        "val_loss": round(val_loss, 5),
        "val_ppl": round(math.exp(min(val_loss, 20)), 3),
        "val_batches": len(losses),
        "val_tokens": n_tokens,
        "val_time_sec": round(time.time() - t0, 2),
    }


def save_checkpoint(
    path: str | Path,
    model: ToyKimiLinear,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LambdaLR,
    cfg: ToyKimiConfig,
    optimizer_step: int,
    micro_step: int,
    best_val_loss_for_stopping: float,
    extra: Optional[dict] = None,
) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "optimizer_step": optimizer_step,
        "micro_step": micro_step,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "config": asdict(cfg),
        "best_val_loss_for_stopping": best_val_loss_for_stopping,
    }
    if extra:
        payload.update(extra)
    torch.save(payload, path)
    print(f"saved checkpoint: {path}")


@dataclass
class EarlyStoppingState:
    patience: int
    min_delta: float
    warmup_evals: int = 0
    best_val_loss: float = float("inf")
    best_step: int = 0
    bad_evals: int = 0
    eval_count: int = 0

    def update(self, val_loss: float, step: int) -> tuple[bool, bool]:
        self.eval_count += 1
        improved = val_loss < (self.best_val_loss - self.min_delta)
        if improved:
            self.best_val_loss = val_loss
            self.best_step = step
            self.bad_evals = 0
        else:
            if self.eval_count > self.warmup_evals:
                self.bad_evals += 1
        return improved, self.bad_evals >= self.patience


def training_status(recent: list[float], best_for_stopping: float, current: float, min_rel: float) -> dict:
    if len(recent) < 4:
        return {"status": "too_early", "reason": "Need at least 4 validation points."}
    rel = (recent[0] - recent[-1]) / max(abs(recent[0]), 1e-8)
    if current > best_for_stopping * 1.01:
        return {"status": "overfitting_or_regressing", "relative_improvement_window": rel}
    if rel < min_rel:
        return {"status": "plateaued", "relative_improvement_window": rel}
    return {"status": "still_learning", "relative_improvement_window": rel}

In [ ]:
# Training

def train(args: SimpleNamespace) -> None:
    if args.run_correctness_tests:
        run_correctness_tests()

    if torch.cuda.is_available():
        device = torch.device("cuda")
        amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        use_amp = True
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
        amp_dtype = torch.float32
        use_amp = False
    else:
        device = torch.device("cpu")
        amp_dtype = torch.float32
        use_amp = False
    print(f"Device: {device} | amp_dtype: {amp_dtype} | use_amp: {use_amp}")

    factory = {
        "quarter": ToyKimiConfig.quarter,
        "half": ToyKimiConfig.half,
        "full": ToyKimiConfig.full,
    }[args.model_size]
    cfg = factory(max_seq_len=args.seq_len, chunk_size=args.chunk_size, dropout=args.dropout)
    cfg.validate()

    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained("gpt2", model_max_length=1_000_000)

    print("Loading data and building splits...")
    token_splits, split_info = make_data_splits(
        dataset_name=args.dataset,
        tokenizer=tokenizer,
        max_tokens=args.max_tokens,
        val_monitor_fraction=args.val_monitor_fraction,
        val_ablation_fraction=args.val_ablation_fraction,
        test_replication_fraction=args.test_replication_fraction,
        val_monitor_max_tokens=args.val_monitor_max_tokens,
        val_ablation_max_tokens=args.val_ablation_max_tokens,
        test_replication_max_tokens=args.test_replication_max_tokens,
    )
    split_checks = verify_splits(token_splits, cfg.max_seq_len)

    datasets = {name: PackedTokenDataset(tokens, cfg.max_seq_len) for name, tokens in token_splits.items()}
    train_loader = DataLoader(
        datasets["train"],
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        pin_memory=(device.type == "cuda"),
        drop_last=True,
    )

    def eval_loader(name: str) -> DataLoader:
        return DataLoader(
            datasets[name],
            batch_size=args.eval_batch_size or args.batch_size,
            shuffle=False,
            num_workers=args.num_workers,
            pin_memory=(device.type == "cuda"),
            drop_last=False,
        )

    val_monitor_loader = eval_loader("val_monitor")
    val_ablation_loader = eval_loader("val_ablation")
    test_replication_loader = eval_loader("test_replication")

    model = ToyKimiLinear(cfg).to(device)
    n_params = model.num_parameters()
    print(f"Model parameters: {n_params:,} ({n_params / 1e6:.2f}M)")
    print(
        f"Samples: train={len(datasets['train']):,}, val_monitor={len(datasets['val_monitor']):,}, "
        f"val_ablation={len(datasets['val_ablation']):,}, test_replication={len(datasets['test_replication']):,}"
    )
    print(f"Split info: {split_info}")
    print(f"Split checks: {split_checks}")

    optimizer = make_optimizer(model, args.lr, args.weight_decay)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda s: cosine_with_warmup(s, args.warmup_steps, args.max_steps),
    )
    use_scaler = device.type == "cuda" and amp_dtype == torch.float16
    scaler = torch.amp.GradScaler(device.type, enabled=use_scaler)

    effective_batch = args.batch_size * args.grad_accum
    tokens_per_step = effective_batch * cfg.max_seq_len
    flops = estimate_flops_per_optimizer_step(cfg, effective_batch)

    run_name = f"toy_kimi_linear_{args.model_size}_{args.dataset}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    log_path = Path(args.log_dir) / f"{run_name}.jsonl"
    checkpoint_dir = Path(args.save_dir) / run_name

    run_config = {
        "arch_name": "toy_kimi_linear_dense_kda_nope_mha",
        "model_config": asdict(cfg),
        "n_params": n_params,
        "dataset": args.dataset,
        "split_info": split_info,
        "split_checks": split_checks,
        "batch_size": args.batch_size,
        "grad_accum": args.grad_accum,
        "effective_batch": effective_batch,
        "tokens_per_optimizer_step": tokens_per_step,
        "lr": args.lr,
        "weight_decay": args.weight_decay,
        "max_steps": args.max_steps,
        "estimated_flops": flops,
        "device": str(device),
        "amp_dtype": str(amp_dtype),
        "use_amp": use_amp,
        "note": "MLA is approximated by NoPE full MHA; MoE is replaced by dense SwiGLU.",
    }
    if device.type == "cuda":
        run_config["gpu_name"] = torch.cuda.get_device_name(device)
        run_config["gpu_memory_gb"] = round(torch.cuda.get_device_properties(device).total_memory / 1e9, 1)

    logger = TrainingLogger(str(log_path), run_config)
    print(f"Logging to: {log_path}")

    early = EarlyStoppingState(
        patience=args.early_stopping_patience,
        min_delta=args.early_stopping_min_delta,
        warmup_evals=args.early_stopping_warmup_evals,
    )
    # Separate numerical-best checkpoint from meaningful early-stopping delta.
    best_checkpoint_loss = float("inf")
    best_checkpoint_step = 0
    recent_val_losses: list[float] = []

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    model.train()
    optimizer.zero_grad(set_to_none=True)
    micro_step = 0
    optimizer_step = 0
    tokens_seen = 0
    running_loss = 0.0
    running_batches = 0
    cumulative_flops = 0.0
    t_start = time.time()
    should_stop = False

    print("\nStarting training")
    print(f"Max optimizer steps: {args.max_steps:,}")
    print(f"Eval every: {args.eval_every:,} optimizer steps")

    while optimizer_step < args.max_steps and not should_stop:
        for x, y in train_loader:
            if optimizer_step >= args.max_steps or should_stop:
                break
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            micro_step += 1
            tokens_seen += x.numel()
            step_t0 = time.time()

            with torch.amp.autocast(device.type, dtype=amp_dtype, enabled=use_amp):
                _, loss, _ = model(x, y)
                loss_backward = loss / args.grad_accum
            scaler.scale(loss_backward).backward()

            running_loss += float(loss.item())
            running_batches += 1

            if micro_step % args.grad_accum == 0:
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip).item()
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

                optimizer_step += 1
                cumulative_flops += flops["estimated_flops_per_optimizer_step"]

                if optimizer_step % args.log_every == 0:
                    elapsed = time.time() - t_start
                    avg_loss = running_loss / max(running_batches, 1)
                    ppl = math.exp(min(avg_loss, 20))
                    tok_s = tokens_seen / max(elapsed, 1e-8)
                    metrics = {
                        "optimizer_step": optimizer_step,
                        "micro_step": micro_step,
                        "train_loss": round(avg_loss, 5),
                        "train_ppl": round(ppl, 3),
                        "lr": scheduler.get_last_lr()[0],
                        "grad_norm": round(grad_norm, 4),
                        "tokens_seen": tokens_seen,
                        "tokens_per_sec": round(tok_s),
                        "step_time_ms": round((time.time() - step_t0) * 1000, 1),
                        "elapsed_min": round(elapsed / 60, 2),
                        "estimated_tflops": round(cumulative_flops / max(elapsed, 1e-8) / 1e12, 2),
                        "dataset_epochs_by_tokens": round(tokens_seen / len(token_splits["train"]), 4),
                        **gpu_stats(device),
                    }
                    logger.log_step(metrics)
                    print(
                        f"step {optimizer_step:>6d}/{args.max_steps} | "
                        f"train_loss {avg_loss:.4f} | train_ppl {ppl:.2f} | "
                        f"lr {scheduler.get_last_lr()[0]:.2e} | tok/s {tok_s:,.0f} | grad {grad_norm:.3f}"
                    )
                    running_loss = 0.0
                    running_batches = 0

                if optimizer_step % args.eval_every == 0 or optimizer_step == 1:
                    eval_metrics = evaluate(
                        model,
                        val_monitor_loader,
                        device,
                        amp_dtype,
                        args.eval_batches,
                        use_amp,
                    )
                    val_loss = float(eval_metrics["val_loss"])
                    recent_val_losses.append(val_loss)
                    recent_val_losses = recent_val_losses[-args.plateau_window :]
                    improved_for_stopping, should_stop = early.update(val_loss, optimizer_step)

                    checkpoint_improved = val_loss < best_checkpoint_loss
                    if checkpoint_improved:
                        best_checkpoint_loss = val_loss
                        best_checkpoint_step = optimizer_step

                    status = training_status(
                        recent_val_losses,
                        early.best_val_loss,
                        val_loss,
                        args.plateau_min_relative_improvement,
                    )
                    payload = {
                        "optimizer_step": optimizer_step,
                        **eval_metrics,
                        "best_checkpoint_loss": round(best_checkpoint_loss, 5),
                        "best_checkpoint_step": best_checkpoint_step,
                        "best_stopping_loss": round(early.best_val_loss, 5),
                        "best_stopping_step": early.best_step,
                        "bad_evals": early.bad_evals,
                        "improved_for_stopping": improved_for_stopping,
                        "checkpoint_improved": checkpoint_improved,
                        "training_status": status,
                    }
                    logger.log_eval(payload)
                    print(
                        f"EVAL step {optimizer_step:>6d} | val_loss {val_loss:.5f} | "
                        f"val_ppl {eval_metrics['val_ppl']:.2f} | "
                        f"best_ckpt {best_checkpoint_loss:.5f} @ step {best_checkpoint_step} | "
                        f"status {status['status']}"
                    )

                    if checkpoint_improved:
                        save_checkpoint(
                            checkpoint_dir / "best.pt",
                            model,
                            optimizer,
                            scheduler,
                            cfg,
                            optimizer_step,
                            micro_step,
                            early.best_val_loss,
                            extra={"eval_metrics": payload},
                        )
                    if should_stop:
                        print(
                            f"Early stopping triggered: validation loss did not improve by "
                            f"at least {args.early_stopping_min_delta} for "
                            f"{args.early_stopping_patience} evals."
                        )
                        break

                if args.save_every > 0 and optimizer_step % args.save_every == 0:
                    save_checkpoint(
                        checkpoint_dir / f"step_{optimizer_step}.pt",
                        model,
                        optimizer,
                        scheduler,
                        cfg,
                        optimizer_step,
                        micro_step,
                        early.best_val_loss,
                    )

    total_time = time.time() - t_start

    # These are included for convenience, but for strict methodology
    ablation_metrics = evaluate(model, val_ablation_loader, device, amp_dtype, args.final_eval_batches, use_amp)
    test_metrics = evaluate(model, test_replication_loader, device, amp_dtype, args.final_eval_batches, use_amp)

    final_metrics = {
        "optimizer_steps": optimizer_step,
        "micro_steps": micro_step,
        "total_time_sec": round(total_time, 1),
        "total_time_hours": round(total_time / 3600, 3),
        "tokens_seen": tokens_seen,
        "avg_tokens_per_sec": round(tokens_seen / max(total_time, 1e-8)),
        "best_checkpoint_loss": round(best_checkpoint_loss, 5),
        "best_checkpoint_step": best_checkpoint_step,
        "best_stopping_loss": round(early.best_val_loss, 5),
        "best_stopping_step": early.best_step,
        "stopped_by_early_stopping": should_stop,
        "val_ablation_metrics": ablation_metrics,
        "test_replication_metrics": test_metrics,
        **gpu_stats(device),
    }
    logger.log_final(final_metrics)
    logger.close()

    save_checkpoint(
        checkpoint_dir / "last.pt",
        model,
        optimizer,
        scheduler,
        cfg,
        optimizer_step,
        micro_step,
        early.best_val_loss,
    )

    print("\nTraining complete")
    print(json.dumps(final_metrics, indent=2))

    print("\nSample generation")
    model.eval()
    prompt = "Once upon a time"
    prompt_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    out_ids = model.generate(
        prompt_ids,
        max_new_tokens=100,
        temperature=0.8,
        top_k=40,
        use_recurrent=(args.sample_generation_mode == "recurrent"),
        optimized_prefill=(not args.no_optimized_prefill),
    )
    print(tokenizer.decode(out_ids[0].tolist()))

In [ ]:
# Colab config


def make_colab_config() -> SimpleNamespace:

    # Model / data
    MODEL_SIZE = "half"             # "quarter", "half", "full"
    DATASET = "tinystories"
    MAX_TOKENS = None               # e.g. 2_000_000 for quick smoke run

    # Splits
    VAL_MONITOR_FRACTION = 0.02
    VAL_ABLATION_FRACTION = 0.015
    TEST_REPLICATION_FRACTION = 0.015
    VAL_MONITOR_MAX_TOKENS = 2_000_000
    VAL_ABLATION_MAX_TOKENS = 1_000_000
    TEST_REPLICATION_MAX_TOKENS = 1_000_000

    # Sequence / batch
    SEQ_LEN = 1024
    CHUNK_SIZE = 64
    BATCH_SIZE = 24
    EVAL_BATCH_SIZE = None
    GRAD_ACCUM = 1
    NUM_WORKERS = 2

    # Optimizer
    LR = 3e-4
    WEIGHT_DECAY = 0.01
    DROPOUT = 0.0
    WARMUP_STEPS = 1000
    MAX_STEPS = 100000
    GRAD_CLIP = 1.0

    # Logging / eval / checkpointing
    LOG_EVERY = 100
    EVAL_EVERY = 500
    EVAL_BATCHES = 50
    FINAL_EVAL_BATCHES = 200
    SAVE_EVERY = 5000
    SAVE_DIR = "/content/drive/MyDrive/models/kimi_linear_toy/checkpoints"
    LOG_DIR = "/content/drive/MyDrive/models/kimi_linear_toy/logs"

    # Early stopping
    EARLY_STOPPING_PATIENCE = 8
    EARLY_STOPPING_MIN_DELTA = 0.002
    EARLY_STOPPING_WARMUP_EVALS = 3
    PLATEAU_WINDOW = 5
    PLATEAU_MIN_RELATIVE_IMPROVEMENT = 0.002

    # Correctness / generation
    RUN_CORRECTNESS_TESTS = True
    SAMPLE_GENERATION_MODE = "recurrent"  # "recurrent" or "full_prefix"
    NO_OPTIMIZED_PREFILL = False

    return SimpleNamespace(
        model_size=MODEL_SIZE,
        dataset=DATASET,
        max_tokens=MAX_TOKENS,
        val_monitor_fraction=VAL_MONITOR_FRACTION,
        val_ablation_fraction=VAL_ABLATION_FRACTION,
        test_replication_fraction=TEST_REPLICATION_FRACTION,
        val_monitor_max_tokens=VAL_MONITOR_MAX_TOKENS,
        val_ablation_max_tokens=VAL_ABLATION_MAX_TOKENS,
        test_replication_max_tokens=TEST_REPLICATION_MAX_TOKENS,
        seq_len=SEQ_LEN,
        chunk_size=CHUNK_SIZE,
        batch_size=BATCH_SIZE,
        eval_batch_size=EVAL_BATCH_SIZE,
        grad_accum=GRAD_ACCUM,
        num_workers=NUM_WORKERS,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        dropout=DROPOUT,
        warmup_steps=WARMUP_STEPS,
        max_steps=MAX_STEPS,
        grad_clip=GRAD_CLIP,
        log_every=LOG_EVERY,
        eval_every=EVAL_EVERY,
        eval_batches=EVAL_BATCHES,
        final_eval_batches=FINAL_EVAL_BATCHES,
        save_every=SAVE_EVERY,
        save_dir=SAVE_DIR,
        log_dir=LOG_DIR,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
        early_stopping_warmup_evals=EARLY_STOPPING_WARMUP_EVALS,
        plateau_window=PLATEAU_WINDOW,
        plateau_min_relative_improvement=PLATEAU_MIN_RELATIVE_IMPROVEMENT,
        run_correctness_tests=RUN_CORRECTNESS_TESTS,
        sample_generation_mode=SAMPLE_GENERATION_MODE,
        no_optimized_prefill=NO_OPTIMIZED_PREFILL,
    )

In [ ]:
train(make_colab_config())

KDA chunkwise/recurrent correctness test passed.
Model step-by-step recurrent path matches full forward test passed.
Optimized prefill cache test passed.
Device: cuda | amp_dtype: torch.bfloat16 | use_amp: True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Model parameters: 27,285,714 (27.29M)
Samples: train=456,906, val_monitor=1,953, val_ablation=976, test_replication=976
Split info: {'dataset': 'tinystories', 'fractions': {'val_monitor': 0.02, 'val_ablation': 0.015, 'test_replication': 0.015}, 'split_policy': 'tail holdout from train token stream', 'n_total_loaded_tokens': 471872517, 'n_tokens': {'train': 467872517, 'val_monitor': 2000000, 'val_ablation': 1000000, 'test_replication': 1000000}}
Split checks: {'train': {'tokens': 467872517, 'samples': 456906}, 'val_monitor': {'tokens': 2000000, 'samples': 1953}, 'val_ablation': {'tokens': 1000000, 'samples': 976}, 'test_replication': {'tokens': 1000000, 'samples': 976}, 'windows_cross_boundary': False, 'note': 'Each split gets its own PackedTokenDataset; no packed window crosses split boundaries.'}
Logging to: /content/drive/MyDrive/models/kimi_linear_toy/logs/toy_kimi_linear_half_tinystories_20260501_124445.jsonl

Starting training
Max optimizer steps: 100,000
Eval every: 500 optimizer